In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os, gc, random, warnings
warnings.filterwarnings('ignore')
 
# Clean slate for reproducibility (competition rule 7.2)
for _f in ['/kaggle/working/checkpoint.pth', '/kaggle/working/best_model.pth']:
    if os.path.exists(_f): os.remove(_f)
 
def set_seeds(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED']       = str(seed)
 
set_seeds(42)
print("✅ Imports done")
print("CUDA:", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

✅ Imports done
CUDA: True
GPU : Tesla T4


In [2]:
class Config:
    _BASE = (
        '/kaggle/input/competitions/aisehack-theme-2'
        if os.path.exists('/kaggle/input/competitions/aisehack-theme-2')
        else '/kaggle/input/aisehack-theme-2'
    )
    DATA_ROOT   = _BASE + '/raw'
    TEST_PATH   = _BASE + '/test_in'
    OUTPUT_PATH = '/kaggle/working/preds.npy'
    MODEL_PATH  = '/kaggle/working/best_model.pth'
    CKPT_PATH   = '/kaggle/working/checkpoint.pth'

    _all_months = os.listdir(DATA_ROOT) if os.path.exists(DATA_ROOT) else []
    MONTHS = sorted([m for m in _all_months if not m.endswith('.npy')])

    # 15 vars: 8 met + 7 chemical precursors (NOx,SO2,NH3,NMVOC_e,NMVOC_finn,bio,PM25)
    ALL_AUX = ['q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain',
               'NOx', 'SO2', 'NH3', 'NMVOC_e', 'NMVOC_finn', 'bio', 'PM25']

    PM25_IN  = 10
    AUX_WIN  = 26
    HORIZON  = 16
    STRIDE   = 1

    # ── Model (bigger than score-17 baseline) ──
    FNO_MODES = 7    # was 12  → captures finer spatial frequencies
    FNO_WIDTH = 8   # was 48  → more representational capacity
    NUM_FNO   = 1     # was 4   → deeper spectral processing
    HIDDEN    = 60   # was 96  → wider step decoder

    # ── Hotspot: IGP / North India (rows 68-94, cols 39-80) ──
    HOT_R1, HOT_R2 = 68, 94
    HOT_C1, HOT_C2 = 39, 80
    HOT_WEIGHT     = 1.5   # upweight high-pollution region in loss

    # ── Training ──
    MAX_EPOCHS   = 80
    ES_PATIENCE  = 15      # early stopping — stops when val stops improving
    BATCH_SIZE   = 4
    LR           = 2e-4
    WEIGHT_DECAY = 2e-5
    GRAD_CLIP    = 1.0
    LOSS_ALPHA   = 3.0     # weight for high-PM2.5 events in loss
    PHYSICS_WT   = 0.05    # physics advection loss weight
    VAL_FRAC     = 0.15    # temporal split — last 15% of each month

    SEED   = 123
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
print(f"✅ Config ready")
print(f"Device    : {cfg.DEVICE}")
print(f"Months    : {cfg.MONTHS}")
print(f"FNO       : modes={cfg.FNO_MODES} width={cfg.FNO_WIDTH} blocks={cfg.NUM_FNO}")
print(f"Max epochs: {cfg.MAX_EPOCHS} | ES patience: {cfg.ES_PATIENCE}")


✅ Config ready
Device    : cuda
Months    : ['APRIL_16', 'DEC_16', 'JULY_16', 'OCT_16']
FNO       : modes=7 width=8 blocks=1
Max epochs: 80 | ES patience: 15


In [3]:
print("Computing normalisation stats...")
 
all_pm25_log = []
for month in cfg.MONTHS:
    arr = np.load(os.path.join(cfg.DATA_ROOT, month, 'cpm25.npy')).astype(np.float32)
    all_pm25_log.append(np.log1p(arr))
all_pm25_log = np.concatenate(all_pm25_log)
PM25_LOG_MIN = float(all_pm25_log.min())
PM25_LOG_MAX = float(all_pm25_log.max())
PM25_LOG_RNG = PM25_LOG_MAX - PM25_LOG_MIN
del all_pm25_log; gc.collect()
print(f"  PM2.5 log: {PM25_LOG_MIN:.4f} → {PM25_LOG_MAX:.4f}")
 
FEAT_MINS, FEAT_MAXS = {}, {}
for feat in cfg.ALL_AUX:
    vals = []
    for month in cfg.MONTHS:
        p = os.path.join(cfg.DATA_ROOT, month, f'{feat}.npy')
        if os.path.exists(p):
            vals.append(np.load(p).astype(np.float32))
    if vals:
        v = np.concatenate(vals)
        FEAT_MINS[feat] = float(v.min())
        FEAT_MAXS[feat] = float(v.max())
        del v
    else:
        FEAT_MINS[feat] = 0.0; FEAT_MAXS[feat] = 1.0
    print(f"  {feat}: {FEAT_MINS[feat]:.4f} → {FEAT_MAXS[feat]:.4f}")
 
gc.collect()
print("✅ Stats done")


Computing normalisation stats...
  PM2.5 log: 0.0012 → 7.9553
  q2: 0.0000 → 0.0379
  t2: 226.4993 → 322.6777
  u10: -25.0179 → 29.0259
  v10: -25.6960 → 31.0424
  swdown: 0.0000 → 1287.9806
  pblh: 53.6355 → 6064.9565
  psfc: 48015.4492 → 101961.3125
  rain: 0.0000 → 397.0356
  NOx: 0.0000 → 0.0000
  SO2: 0.0000 → 0.0000
  NH3: 0.0000 → 0.0000
  NMVOC_e: 0.0000 → 0.0000
  NMVOC_finn: 0.0000 → 0.0000
  bio: 0.0000 → 0.0000
  PM25: 0.0000 → 0.0000
✅ Stats done


In [4]:
def norm_pm25(arr):
    return ((np.log1p(arr) - PM25_LOG_MIN) / (PM25_LOG_RNG + 1e-8)).astype(np.float32)
 
def denorm_pm25_np(arr):
    return np.expm1(np.clip(arr * (PM25_LOG_RNG + 1e-8) + PM25_LOG_MIN, 0, None))
 
 
def load_month(month):
    base    = os.path.join(cfg.DATA_ROOT, month)
    arr_raw = np.load(os.path.join(base, 'cpm25.npy')).astype(np.float32)
    T       = arr_raw.shape[0]
    data    = {'cpm25': norm_pm25(arr_raw)}
    for feat in cfg.ALL_AUX:
        p   = os.path.join(base, f'{feat}.npy')
        arr = np.load(p).astype(np.float32) if os.path.exists(p) \
              else np.zeros((T, 140, 124), dtype=np.float32)
        mn, mx     = FEAT_MINS[feat], FEAT_MAXS[feat]
        data[feat] = ((arr - mn) / (mx - mn + 1e-8)).astype(np.float32)
    print(f"  {month}: {T} hours")
    return data
 
 
def get_t0_list_temporal(T):
    """Temporal split — last 15% of each month is val, no leakage."""
    total   = cfg.AUX_WIN + cfg.HORIZON          # 42 timesteps needed per sample
    all_idx = list(range(0, T - total + 1, cfg.STRIDE))
    split   = int(len(all_idx) * (1 - cfg.VAL_FRAC))
    return all_idx[:split], all_idx[split:]
 
 
class PM25Dataset(Dataset):
    def __init__(self, month_arrays, indices):
        self.data    = month_arrays
        self.indices = indices
        self.u_idx   = cfg.ALL_AUX.index('u10')
        self.v_idx   = cfg.ALL_AUX.index('v10')
 
    def __len__(self): return len(self.indices)
 
    def __getitem__(self, i):
        m, t0           = self.indices[i]
        pm25, aux_stack = self.data[m]
        x_pm  = pm25[t0 : t0 + cfg.PM25_IN]
        x_aux = aux_stack[t0 : t0 + cfg.AUX_WIN]
        y     = pm25[t0 + cfg.PM25_IN : t0 + cfg.PM25_IN + cfg.HORIZON]
        fut_u = aux_stack[t0 + cfg.PM25_IN : t0 + cfg.AUX_WIN, self.u_idx]
        fut_v = aux_stack[t0 + cfg.PM25_IN : t0 + cfg.AUX_WIN, self.v_idx]
        return (torch.from_numpy(x_pm.copy()),
                torch.from_numpy(x_aux.copy()),
                torch.from_numpy(y.copy()),
                torch.from_numpy(fut_u.copy()),
                torch.from_numpy(fut_v.copy()))
 
print("✅ Data pipeline ready")


✅ Data pipeline ready


In [5]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1 = modes1; self.modes2 = modes2
        scale        = 1 / (in_ch * out_ch)
        self.weights1 = nn.Parameter(scale * torch.rand(in_ch, out_ch, modes1, modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(scale * torch.rand(in_ch, out_ch, modes1, modes2, dtype=torch.cfloat))
 
    def compl_mul2d(self, x, w):
        return torch.einsum('bixy,ioxy->boxy', x, w)
 
    def forward(self, x):
        B, C, H, W = x.shape
        x_ft   = torch.fft.rfft2(x.float())
        out_ft = torch.zeros(B, self.weights1.shape[1], H, W//2+1,
                             device=x.device, dtype=torch.cfloat)
        out_ft[:, :, :self.modes1, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)
        return torch.fft.irfft2(out_ft, s=(H, W))
 
 
class FNOBlock(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.spec = SpectralConv2d(width, width, modes, modes)
        self.conv = nn.Conv2d(width, width, 1)
        self.norm = nn.GroupNorm(8, width)   # 24%8=0 ✅
 
    def forward(self, x):
        return F.gelu(self.norm(self.spec(x) + self.conv(x)))
 
 
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        # out_ch values used: 64, 128, 256 — all divisible by 8 ✅
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8, out_ch),
            nn.GELU(),
            nn.Dropout2d(0.1),               # ← only addition vs 14.778 model
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8, out_ch),
            nn.GELU(),
        )
    def forward(self, x): return self.block(x)
 
 
class PTM_FNO(nn.Module):
    def __init__(self):
        super().__init__()
        W    = cfg.FNO_WIDTH          # 24
        NAUX = len(cfg.ALL_AUX)       # 8
        # in_ch = 10 + 26*8 = 218
        self.lift = nn.Sequential(
            nn.Conv2d(cfg.PM25_IN + cfg.AUX_WIN * NAUX, W * 2, 1), nn.GELU(),
            nn.Conv2d(W * 2, W, 1),
        )
        self.fno_blocks = nn.Sequential(
            *[FNOBlock(W, cfg.FNO_MODES) for _ in range(cfg.NUM_FNO)]
        )
        # U-Net
        self.enc1 = DoubleConv(W,       64)   # GroupNorm(8,64)  ✅
        self.enc2 = DoubleConv(64,     128)   # GroupNorm(8,128) ✅
        self.bot  = DoubleConv(128,    256)   # GroupNorm(8,256) ✅
        self.dec2 = DoubleConv(256+128,128)   # GroupNorm(8,128) ✅
        self.dec1 = DoubleConv(128+64,  64)   # GroupNorm(8,64)  ✅
        self.fuse = nn.Conv2d(W + 64, W, 1)
        self.pool = nn.MaxPool2d(2, ceil_mode=True)
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
 
        # AR step decoder — input: W(24) + NAUX(8) + 1 = 33
        self.step_conv = nn.Sequential(
            nn.Conv2d(W + NAUX + 1, cfg.HIDDEN, 3, padding=1), nn.GELU(),
            nn.Conv2d(cfg.HIDDEN, cfg.HIDDEN,   3, padding=1), nn.GELU(),
            nn.Conv2d(cfg.HIDDEN, 1, 1),
        )
        # Direct head — all 16 steps at once, no error accumulation
        self.direct_head = nn.Sequential(
            nn.Conv2d(W, cfg.HIDDEN, 3, padding=1), nn.GELU(),
            nn.Conv2d(cfg.HIDDEN, cfg.HORIZON, 1),
        )
 
    def encode(self, pm25_hist, aux):
        B        = pm25_hist.shape[0]
        aux_flat = aux.view(B, cfg.AUX_WIN * len(cfg.ALL_AUX), 140, 124)
        x        = self.lift(torch.cat([pm25_hist, aux_flat], 1))
        fno_out  = self.fno_blocks(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b  = self.bot(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up(b),  e2], 1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], 1))
        return F.gelu(self.fuse(torch.cat([fno_out, d1], 1)))
 
    def forward(self, pm25_hist, aux):
        state = self.encode(pm25_hist, aux)
 
        # AR path
        cur   = pm25_hist[:, -1:, :, :]
        preds = []
        for t in range(cfg.HORIZON):
            forcing = aux[:, cfg.PM25_IN + t, :, :]   # shape (B, 8, H, W)
            delta   = self.step_conv(torch.cat([state, forcing, cur], 1))
            cur     = cur + delta
            preds.append(cur)
        ar_out = torch.cat(preds, 1)          # (B, 16, H, W)
 
        # Direct path
        direct_out = self.direct_head(state)  # (B, 16, H, W)
 
        # 60% AR + 40% direct
        return 0.6 * ar_out + 0.4 * direct_out
 
 
# ── Smoke test ────────────────────────────────────────────────
_m = PTM_FNO().to(cfg.DEVICE)
_p = torch.randn(1, cfg.PM25_IN, 140, 124).to(cfg.DEVICE)
_a = torch.randn(1, cfg.AUX_WIN, len(cfg.ALL_AUX), 140, 124).to(cfg.DEVICE)
_o = _m(_p, _a)
assert _o.shape == (1, cfg.HORIZON, 140, 124), f"Shape error: {_o.shape}"
print(f"✅ PTM_FNO ready | Params: {sum(p.numel() for p in _m.parameters()):,}")
print(f"   in_ch={cfg.PM25_IN + cfg.AUX_WIN * len(cfg.ALL_AUX)}  step_conv_in={cfg.FNO_WIDTH + len(cfg.ALL_AUX) + 1}")
del _m, _p, _a, _o

✅ PTM_FNO ready | Params: 1,952,905
   in_ch=400  step_conv_in=24


In [6]:
_hot_mask = torch.ones(1, 1, 140, 124)
_hot_mask[:, :, cfg.HOT_R1:cfg.HOT_R2, cfg.HOT_C1:cfg.HOT_C2] = cfg.HOT_WEIGHT
HOT_MASK = _hot_mask.to(cfg.DEVICE)
 
 
def weighted_rmse_loss(pred, target):
    mean_pm = target.mean(dim=(2, 3), keepdim=True).detach() + 1e-6
    weight  = (1.0 + cfg.LOSS_ALPHA * (target / mean_pm)) * HOT_MASK
    return (weight * (pred - target) ** 2).mean()
 
 
def physics_loss(pred, wind_u, wind_v):
    dpm_dx = pred[:, :, :, 1:] - pred[:, :, :, :-1]
    dpm_dy = pred[:, :, 1:, :] - pred[:, :, :-1, :]
    u = wind_u[:, :, :, :dpm_dx.shape[3]]
    v = wind_v[:, :, :dpm_dy.shape[2], :]
    return ((dpm_dx * u.sign()).clamp(max=0).abs().mean() +
            (dpm_dy * v.sign()).clamp(max=0).abs().mean())
 
 
def total_loss(pred, target, wu, wv):
    return weighted_rmse_loss(pred, target) + cfg.PHYSICS_WT * physics_loss(pred, wu, wv)
 
 
def val_metric_real(pred_norm, target_norm):
    """Exact leaderboard metric: spatial RMSE averaged in real µg/m³."""
    p = denorm_pm25_np(pred_norm.detach().cpu().float().numpy())
    t = denorm_pm25_np(target_norm.detach().cpu().float().numpy())
    return float(np.mean(np.sqrt(np.mean((p - t) ** 2, axis=(2, 3)))))
 
 
print("✅ Loss & metric ready")


✅ Loss & metric ready


In [7]:
print("Loading training data...")
month_arrays  = []
train_indices = []
val_indices   = []
 
for month in cfg.MONTHS:
    mpath = os.path.join(cfg.DATA_ROOT, month)
    if not os.path.exists(mpath):
        print(f"  Skipping {month}"); continue
    print(f"\n  {month}...")
    d    = load_month(month)
    pm25 = d['cpm25']
    aux  = np.stack([d[f] for f in cfg.ALL_AUX], axis=1)  # (T, 8, H, W)
    midx = len(month_arrays)
    month_arrays.append((pm25, aux))
    tr, va = get_t0_list_temporal(pm25.shape[0])
    train_indices.extend([(midx, t) for t in tr])
    val_indices.extend(  [(midx, t) for t in va])
    print(f"    train={len(tr)}  val={len(va)}")
 
train_ds     = PM25Dataset(month_arrays, train_indices)
val_ds       = PM25Dataset(month_arrays, val_indices)
train_loader = DataLoader(train_ds, cfg.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   cfg.BATCH_SIZE * 2, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f"\n✅ Train: {len(train_ds)} | Val: {len(val_ds)}")


Loading training data...

  APRIL_16...
  APRIL_16: 715 hours
    train=572  val=102

  DEC_16...
  DEC_16: 739 hours
    train=593  val=105

  JULY_16...
  JULY_16: 739 hours
    train=593  val=105

  OCT_16...
  OCT_16: 739 hours
    train=593  val=105

✅ Train: 2351 | Val: 417


In [8]:
from tqdm.notebook import tqdm
 
model     = PTM_FNO().to(cfg.DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.MAX_EPOCHS, eta_min=1e-6)
 
best_val    = float('inf')
best_state  = None
start_epoch = 1
es_counter  = 0
 
if os.path.exists(cfg.CKPT_PATH):
    try:
        ck = torch.load(cfg.CKPT_PATH, map_location=cfg.DEVICE)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        best_val    = ck['best_val']
        best_state  = ck['best_state']
        start_epoch = ck['epoch'] + 1
        es_counter  = ck.get('es_counter', 0)
        print(f"✅ Resumed epoch {ck['epoch']} | best: {best_val:.2f} µg/m³ | patience: {es_counter}/{cfg.ES_PATIENCE}")
    except Exception as e:
        print(f"⚠️ Bad checkpoint, fresh start ({e})")
        os.remove(cfg.CKPT_PATH)
else:
    print(f"Fresh start | max={cfg.MAX_EPOCHS} epochs | ES patience={cfg.ES_PATIENCE}")
 
ebar = tqdm(range(start_epoch, cfg.MAX_EPOCHS + 1), desc='Training', unit='ep',
            bar_format='{l_bar}{bar:30}{r_bar}')
 
for epoch in ebar:
    # ── Train ────────────────────────────────────────────────
    model.train()
    tr_losses = []
    bbar = tqdm(train_loader, desc=f'  E{epoch:03d}[Trn]',
                leave=False, bar_format='{l_bar}{bar:20}{r_bar}')
    for pm25, aux, tgt, wu, wv in bbar:
        pm25 = pm25.to(cfg.DEVICE); aux  = aux.to(cfg.DEVICE)
        tgt  = tgt.to(cfg.DEVICE);  wu   = wu.to(cfg.DEVICE); wv = wv.to(cfg.DEVICE)
        optimizer.zero_grad()
        pred = model(pm25, aux)
        loss = total_loss(pred, tgt, wu, wv)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
        optimizer.step()
        tr_losses.append(loss.item())
        bbar.set_postfix(loss=f'{loss.item():.4f}')
    bbar.close()
    scheduler.step()
 
    # ── Validate — real µg/m³ matches leaderboard ────────────
    model.eval()
    val_scores = []
    with torch.no_grad():
        for pm25, aux, tgt, wu, wv in tqdm(val_loader, desc=f'  E{epoch:03d}[Val]',
                                            leave=False, bar_format='{l_bar}{bar:20}{r_bar}'):
            pred = model(pm25.to(cfg.DEVICE), aux.to(cfg.DEVICE))
            val_scores.append(val_metric_real(pred, tgt))
 
    mean_val = float(np.mean(val_scores))
    mean_tr  = float(np.mean(tr_losses))
 
    # ── Early stopping ────────────────────────────────────────
    if mean_val < best_val:
        best_val   = mean_val
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        es_counter = 0
        tqdm.write(f'  🏆 E{epoch:03d} | tr={mean_tr:.4f} val={best_val:.2f} µg/m³  ← BEST')
    else:
        es_counter += 1
        tqdm.write(f'  ⏳ E{epoch:03d} | tr={mean_tr:.4f} val={mean_val:.2f} µg/m³  [{es_counter}/{cfg.ES_PATIENCE}]')
 
    ebar.set_postfix(val=f'{mean_val:.2f}µg', best=f'{best_val:.2f}µg',
                     pat=f'{es_counter}/{cfg.ES_PATIENCE}')
 
    if epoch % 5 == 0:
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                    'best_val': best_val, 'best_state': best_state,
                    'es_counter': es_counter}, cfg.CKPT_PATH)
        tqdm.write(f'  💾 Checkpoint epoch {epoch}')
 
    if es_counter >= cfg.ES_PATIENCE:
        tqdm.write(f'\n🛑 Early stopping at epoch {epoch} | Best: {best_val:.2f} µg/m³')
        break
 
ebar.close()
model.load_state_dict(best_state)
torch.save(best_state, cfg.MODEL_PATH)
print(f"\n✅ Done! Best val: {best_val:.2f} µg/m³ | Stopped at epoch {epoch}/{cfg.MAX_EPOCHS}")
print(f"   Expected leaderboard: ~{best_val*1.05:.1f} - {best_val*1.15:.1f} µg/m³")


Fresh start | max=80 epochs | ES patience=15


Training:   0%|                              | 0/80 [00:00<?, ?ep/s]

  E001[Trn]:   0%|                    | 0/587 [00:00<?, ?it/s]

  E001[Val]:   0%|                    | 0/53 [00:00<?, ?it/s]

  🏆 E001 | tr=0.0204 val=17.70 µg/m³  ← BEST


...

  E079[Val]:   0%|                    | 0/53 [00:00<?, ?it/s]

  ⏳ E079 | tr=0.0019 val=11.67 µg/m³  [12/15]


  E080[Trn]:   0%|                    | 0/587 [00:00<?, ?it/s]

  E080[Val]:   0%|                    | 0/53 [00:00<?, ?it/s]

  ⏳ E080 | tr=0.0019 val=11.66 µg/m³  [13/15]
  💾 Checkpoint epoch 80

✅ Done! Best val: 11.59 µg/m³ | Stopped at epoch 80/80
   Expected leaderboard: ~12.2 - 13.3 µg/m³


In [9]:
del month_arrays, train_ds, val_ds, train_loader, val_loader
del train_indices, val_indices
torch.cuda.empty_cache(); gc.collect()
import psutil
print(f"✅ RAM cleared | Free: {psutil.virtual_memory().available/1024**3:.1f} GB")

✅ RAM cleared | Free: 23.5 GB


In [10]:
gc.collect(); torch.cuda.empty_cache()
 
N, BATCH = 996, 8
 
model = PTM_FNO().to(cfg.DEVICE)
model.load_state_dict(torch.load(cfg.MODEL_PATH, map_location=cfg.DEVICE))
model.eval()
print("✅ Model loaded")
 
pm25_raw  = np.load(os.path.join(cfg.TEST_PATH, 'cpm25.npy')).astype(np.float32)
pm25_norm = norm_pm25(pm25_raw)
del pm25_raw; gc.collect()
print(f"✅ Test PM2.5: {pm25_norm.shape}")   # expect (996, 10, 140, 124)
 
aux_mmap = {feat: np.load(os.path.join(cfg.TEST_PATH, f'{feat}.npy'), mmap_mode='r')
            for feat in cfg.ALL_AUX}
print(f"✅ Aux features loaded: {len(aux_mmap)}")   # expect 8
 
all_preds = []
for i in range(0, N, BATCH):
    aux_list = []
    for feat in cfg.ALL_AUX:
        arr = aux_mmap[feat][i:i+BATCH].astype(np.float32)
        aux_list.append((arr - FEAT_MINS[feat]) / (FEAT_MAXS[feat] - FEAT_MINS[feat] + 1e-8))
    aux_b  = np.stack(aux_list, axis=2).astype(np.float32)   # (B, 26, 8, H, W)
    pm25_t = torch.from_numpy(pm25_norm[i:i+BATCH]).float().to(cfg.DEVICE)
    aux_t  = torch.from_numpy(aux_b).float().to(cfg.DEVICE)
    del aux_b, aux_list
    with torch.no_grad():
        pred_b = model(pm25_t, aux_t)
    all_preds.append(pred_b.cpu().float().numpy())
    del pm25_t, aux_t, pred_b
    torch.cuda.empty_cache()
    if i % 200 == 0: print(f"  {i}/{N}...")
 
preds_norm = np.concatenate(all_preds, axis=0)        # (996, 16, 140, 124)
preds      = np.clip(denorm_pm25_np(preds_norm), 0, 500)
preds      = preds.transpose(0, 2, 3, 1)              # (996, 140, 124, 16)
 
assert preds.shape == (996, 140, 124, 16), f"Shape error: {preds.shape}"
assert np.isfinite(preds).all(), "NaN/Inf in predictions!"
assert preds.min() >= 0, f"Negative values: {preds.min()}"
 
np.save(cfg.OUTPUT_PATH, preds)
print(f"\n✅ Saved → {cfg.OUTPUT_PATH}")
print(f"   Shape={preds.shape}  Min={preds.min():.1f}  Max={preds.max():.1f}"
      f"  Mean={preds.mean():.1f}  Std={preds.std():.1f} µg/m³")
print()
print("HEALTH CHECK:")
print(f"   Mean ~28-36 µg/m³ → good generalisation")
print(f"   Mean  >45  µg/m³ → likely overfit")
print(f"   Max   =500        → clipped (expected)")

✅ Model loaded
✅ Test PM2.5: (996, 10, 140, 124)
✅ Aux features loaded: 15
  0/996...
  200/996...
  400/996...
  600/996...
  800/996...

✅ Saved → /kaggle/working/preds.npy
   Shape=(996, 140, 124, 16)  Min=0.0  Max=500.0  Mean=33.7  Std=36.3 µg/m³

HEALTH CHECK:
   Mean ~28-36 µg/m³ → good generalisation
   Mean  >45  µg/m³ → likely overfit
   Max   =500        → clipped (expected)
